In [ ]:
import torch
# if torch.cuda.is_available():
#     print(f"Number of GPUs available: {torch.cuda.device_count()}")
#     gpu_name = torch.cuda.get_device_name(0)
#     print(f"GPU Name: {gpu_name}")


: 

: 

In [ ]:

import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import matplotlib.pyplot as plt
pl.seed_everything(42)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 42


42

# Dataset

In [ ]:

from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import json
from PIL import Image
from torchvision.transforms import v2


class UGMDataset(Dataset):
    def __init__(self, 
                 root="dataset", 
                 size=1024,
                 split="train",
                 ):
        self.root = Path(root)
        self.size = size
        self.split = split
        with open(self.root / f"{split}.json", 'r') as file:
            self.data = json.load(file)
        self.images = [v for k, v in self.data.items()]

        self.augmentation = v2.Compose([
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
            v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            v2.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
        ])
        
        self.teacher_transform = v2.Compose([
            v2.Resize((1024, 1024)),
            v2.ToTensor(),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.student_transform = v2.Compose([
            v2.Resize((size, size)),
            v2.ToTensor(),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = Image.open(self.root/"images"/f"{self.images[idx]}.png")
        if self.split == "train":
            image = self.augmentation(image)
        
        t_image = self.teacher_transform(image)
        s_image = self.student_transform(image)
        
        return {
            "teacher_image": t_image,
            "student_image": s_image,
            "image_name": self.images[idx],
        }

    def show_pseudo_label(self, image):
        color_map = torch.tensor([
            [128, 64,128], [244, 35,232], [ 70, 70, 70], [102,102,156],
            [190,153,153], [153,153,153], [250,170, 30], [220,220,  0],
            [107,142, 35], [152,251,152], [ 70,130,180], [220, 20, 60],
            [255,  0,  0], [  0,  0,142], [  0,  0, 70], [  0, 60,100],
            [  0, 80,100], [  0,  0,230], [119, 11, 32]  # 19 classes
        ], dtype=torch.uint8)

        h, w = image.shape
        color_image = torch.zeros((h, w, 3), dtype=torch.uint8, device=image.device)
        for c in range(19):
            mask = image == c
            color_image[mask] = color_map[c]
        return color_image.permute(2, 0, 1)


class UGMDataModule(pl.LightningDataModule):
    def __init__(self, 
                 root="dataset", 
                 size=1024,
                 batch_size=8,
                 num_workers=4,
                 ):
        super().__init__()
        self.root = root
        self.size = size
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_data = UGMDataset(root=self.root, size=self.size, split="train")
        self.val_data = UGMDataset(root=self.root, size=self.size, split="val")
        self.test_data = UGMDataset(root=self.root, size=self.size, split="test")

    def train_dataloader(self):
        return DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_data, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True)
    
    def test_dataloader(self):
        return DataLoader(self.test_data, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True)

testing:

In [ ]:

_test = UGMDataModule(batch_size=4, num_workers=4)
_test.setup()
for batch in _test.train_dataloader():
    input_tensor = batch["student_image"]
    print(input_tensor.shape)
    print(input_tensor.device)
    break

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


torch.Size([4, 3, 1024, 1024])
cpu


# Models

### 1. Teacher (Segformer-B5)

In [ ]:
from transformers import SegformerForSemanticSegmentation
teacher_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
teacher_model.config.output_attentions = True
teacher_model.config.output_hidden_states = True

In [ ]:
def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train()  # Set dropout to train mode

def mc_dropout_predict(model, input_tensor, n_forward=30):
    model.eval()
    enable_dropout(model)  # Enable dropout during inference
    preds = []

    with torch.no_grad():
        for _ in range(n_forward):
            output = model(input_tensor)
            preds.append(output.logits.unsqueeze(0))

    preds = torch.cat(preds, dim=0)  # [n_forward, batch, ...]
    mean = preds.mean(dim=0)
    std = preds.std(dim=0)
    return mean, std

### 2. Student (EfficientNet-Unet)

In [ ]:
import segmentation_models_pytorch as smp

class StudentModel(nn.Module):
    def __init__(self, encoder="timm-efficientnet-b0", num_classes=19):
        super().__init__()
        self.model = smp.Unet(
            encoder_name=encoder,
            encoder_weights="imagenet",
            in_channels=3,
            classes=num_classes
        )
        
    def forward(self, x):
        return self.model(x)

# Methods

### 1. Pseudo-Label Generation

In [ ]:
def generate_pseudo_labels(model, dataloader, threshold=0.9, device='cuda'):
    model.eval()
    pseudo_labels = []
    with torch.no_grad():
        for images in dataloader:
            images = images.to(device)
            logits = model(images)  # [B, C, H, W]
            probs = torch.softmax(logits, dim=1)
            conf, pseudo = probs.max(dim=1)  # [B, H, W]
            
            # Mask: only keep high-confidence pixels
            mask = conf > threshold
            pseudo[~mask] = 255
            pseudo_labels.append(pseudo.cpu())
            
    return torch.cat(pseudo_labels)

### 2. PRN: Pseudo-Label Refinement Network

In [ ]:
class PRN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(num_classes+3, 32, 3, padding=1, groups=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1, groups=32, bias=False),  # Depthwise
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(32, num_classes, 1)
    def forward(self, img, teacher_logits):
        x = torch.cat([img, teacher_logits], dim=1)
        x = self.encoder(x)
        return self.head(x)

In [ ]:
def prn_loss(refined_logits, teacher_logits, confident_mask):
    ce_loss = F.cross_entropy(refined_logits, teacher_logits.argmax(1), reduction='none')
    # Only high-confidence pixels supervise PRN directly
    ce_loss = ce_loss * confident_mask.float()
    # KL-divergence for consistency on all pixels
    kl_loss = F.kl_div(
        F.log_softmax(refined_logits, dim=1),
        F.softmax(teacher_logits, dim=1),
        reduction='batchmean'
    )
    return ce_loss.mean() + 0.1 * kl_loss

### 3. Curriculum-Based Pixel Selection

In [ ]:
def curriculum_threshold(epoch, total_epochs, min_thr=0.6, max_thr=0.95):
    from math import cos, pi
    return min_thr + 0.5 * (max_thr - min_thr) * (1 + cos(pi * epoch / total_epochs))

### 4. Negative Sampling from Ambiguous Pixels (U2PL-style)

In [ ]:
def negative_loss(student_logits, teacher_probs, threshold=0.5, topk=3):
    # student_logits: [B, C, H, W], teacher_probs: [B, C, H, W]
    max_probs, _ = teacher_probs.max(dim=1)         # [B, H, W]
    ambiguous = (max_probs < threshold)             # ambiguous pixel mask

    # Get top-K indices per-pixel where confidence is high
    topk_vals, topk_inds = teacher_probs.topk(topk, dim=1)  # [B, topk, H, W]
    # For each ambiguous pixel, penalize these classes
    b, _, h, w = student_logits.shape
    loss = 0
    for k in range(topk):
        mask = ambiguous & (topk_inds[:, k, :, :] != student_logits.argmax(1))
        # Penalize if student predicts one of the top-k as positive
        neg_class = topk_inds[:, k, :, :][mask]
        student_pred = student_logits.permute(0,2,3,1)[mask] # [N, C]
        if neg_class.numel() > 0:
            loss += F.cross_entropy(student_pred, neg_class, reduction='mean')
    return loss / topk

# Training

In [ ]:
from torch.nn import CrossEntropyLoss, KLDivLoss
from segmentation_models_pytorch.losses import DiceLoss
from segmentation_models_pytorch.metrics import get_stats, accuracy, iou_score, f1_score

class BaseModel(pl.LightningModule):
    def __init__(self, student_model, teacher_model, num_classes=19, lr=1e-3, weight_decay=1e-4, **kwargs):
        super().__init__()
        self.save_hyperparameters()
        
        self.student = student_model
        self.teacher = teacher_model
        self.num_classes = num_classes
        self.lr = lr
        self.weight_decay = weight_decay
        self.prn = PRN(num_classes)
        
        self.use_kd = kwargs.get('use_kd', False)
        self.T = kwargs.get('T', 2.0)
        self.kd_weight = kwargs.get('kd_weight', 0.5)
        
    def forward(self, x):
        return self.student(x)
    
    def compute_loss(self, student_logits, teacher_logits):
        ce = CrossEntropyLoss(ignore_index=255)
        dice = DiceLoss(mode='multiclass', ignore_index=255)
        kl = KLDivLoss(reduction='batchmean')
        
        targets = teacher_logits.argmax(dim=1)
        ce_loss = ce(student_logits, targets)
        dice_loss = dice(student_logits, targets)
        seg_loss = ce_loss + dice_loss
        if self.use_kd:
            # Knowledge Distillation Loss
            teacher_probs = F.softmax(teacher_logits / self.T, dim=1)
            student_probs = F.log_softmax(student_logits / self.T, dim=1)
            kd_loss = kl(student_probs, teacher_probs) * (self.T ** 2)
            loss = seg_loss + self.kd_weight * kd_loss
        else:
            loss = seg_loss
            
        return {
            'total': loss,
            'seg': seg_loss,
            'kd': kd_loss if self.use_kd else torch.tensor(0.0),
        }
    
    def compute_metrics(self, preds, targets):
        tp, fp, tn, fn = get_stats(preds, targets, mode='multiclass', num_classes=num_classes, ignore_index=ignore_index)
        
        # Micro: Compute metric across all pixels and classes at once.
        acc_micro = accuracy(tp, fp, tn, fn, reduction='micro')
        iou_micro = iou_score(tp, fp, tn, fn, reduction='micro')
        f1_micro = f1_score(tp, fp, tn, fn, reduction='micro')
        
        # Micro-imagewise: Compute metric for each image, then average over all images.
        acc_micro_imagewise = accuracy(tp, fp, tn, fn, reduction='micro-imagewise')
        iou_micro_imagewise = iou_score(tp, fp, tn, fn, reduction='micro-imagewise')
        f1_micro_imagewise = f1_score(tp, fp, tn, fn, reduction='micro-imagewise')
        
        # Macro: Compute metric per class, then average over all classes.
        acc_macro = accuracy(tp, fp, tn, fn, reduction='macro')
        iou_macro = iou_score(tp, fp, tn, fn, reduction='micro')
        f1_macro = f1_score(tp, fp, tn, fn, reduction='macro')
        
        # Macro-imagewise: Compute metric per class per image, then average over all images and classes.
        acc_macro_imagewise = accuracy(tp, fp, tn, fn, reduction='macro-imagewise')
        iou_macro_imagewise = iou_score(tp, fp, tn, fn, reduction='macro-imagewise')
        f1_macro_imagewise = f1_score(tp, fp, tn, fn, reduction='macro-imagewise')
        
        return {
            'acc': {
                'micro': acc_micro,
                'micro_imagewise': acc_micro_imagewise,
                'macro': acc_macro,
                'macro_imagewise': acc_macro_imagewise
            },
            'iou': {
                'micro': iou_micro,
                'micro_imagewise': iou_micro_imagewise,
                'macro': iou_macro,
                'macro_imagewise': iou_macro_imagewise
            },
            'f1': {
                'micro': f1_micro,
                'micro_imagewise': f1_micro_imagewise,
                'macro': f1_macro,
                'macro_imagewise': f1_macro_imagewise
            }
        }
    
    def step(self, batch, stage):
        t_images = batch['teacher_image']
        s_images = batch['student_image']
        
        t_logits = self.teacher(pixel_values=t_images).logits
        s_logits = self.student(s_images)
        
        loss_dict = self.compute_loss(s_logits, t_logits)
        
        # Pseudo-labeling
        preds = s_logits.argmax(dim=1)
        pseudo_labels = t_logits.argmax(dim=1)
        
        metrics_dict = self.compute_metrics(preds, pseudo_labels)
        
        return {
            "loss": loss_dict,
            "metrics": metrics_dict,
            "image": s_images[0],
            "pseudo_label": pseudo_labels[0],
            "pred": preds[0],
        }
    
    def logging(self, step_output, stage="train"):
        # Log losses
        self.log(
            f"loss/total/{stage}",
            step_output["loss"]["total"],
            on_step=True,
            on_epoch=True,
            prog_bar=True,
        )
        if self.use_kd:
            self.log(
                f"loss/seg/{stage}",
                step_output["loss"]["seg"],
                on_step=True,
                on_epoch=True,
            )
            self.log(
                f"loss/kd/{stage}",
                step_output["loss"]["kd"],
                on_step=True,
                on_epoch=True,
            )

        # Log metrics
        for metric, values in step_output["metrics"].items():
            for sub_metric, value in values.items():
                self.log(
                    f"{metric}/{sub_metric}/{stage}", value, on_step=True, on_epoch=True
                )

        # Log images every 10 steps for non-training stages
        if stage != "train":
            self.logger.experiment.add_image(
                f"{stage}_samples/image",
                step_output["image"],
                self.global_step,
                dataformats="CHW",
            )
            self.logger.experiment.add_image(
                f"{stage}_samples/target",
                colorize_segmentation(step_output["pseudo_label"].cpu()),
                self.global_step,
                dataformats="CHW",
            )
            self.logger.experiment.add_image(
                f"{stage}_samples/prediction",
                colorize_segmentation(step_output["pred"].cpu()),
                self.global_step,
                dataformats="CHW",
            )
    
    def training_step(self, batch, _):
        self.student.train()
        output = self.step(batch, stage='train')
        self.logging(output, stage='train')
        return output["loss"]["total"]
    
    def validation_step(self, batch, _):
        self.student.eval()
        output = self.step(batch, stage='val')
        self.logging(output, stage='val')
        return output
    
    def test_step(self, batch, _):
        self.student.eval()
        output = self.step(batch, stage='test')
        self.logging(output, stage='test')
        return output
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            list(self.student.parameters()) + list(self.prn.parameters()), lr=self.lr, weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.2, patience=2, verbose=True)
        return [optimizer], [{"scheduler": scheduler, "interval": "epoch", "monitor": "loss/total/val"}]

In [ ]:
USE_KD = False
T = 2.0
KD_WEIGHT = 0.001
SIZE = 256
ENCODER = "timm-efficientnet-b0"

In [ ]:
model = BaseModel(
    student_model=StudentModel(encoder=ENCODER),
    teacher_model=teacher_model,
    use_kd=False,
    T=2.0,
    kd_weight=0.001
)

dm = UGMDataModule(
    root="dataset", 
    size=256, 
    batch_size=20, 
    num_workers=4
)

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'student_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['student_model'])`.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'teacher_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_model'])`.


In [ ]:
kd_str = f"_kd_T{T}_w-{KD_WEIGHT}" if USE_KD else ""
version = (
    f"base_batch-{dm.batch_size}"
    f"{kd_str}_{SIZE}x{SIZE}"
)
name = f"{ENCODER}"
logger = pl.loggers.TensorBoardLogger("test_logs/", name=name, version=version)
trainer = pl.Trainer(
    max_epochs=100,
    accelerator="gpu",
    devices=2,
    strategy='ddp_notebook',
    precision=64,
    profiler="simple",
    logger=logger,
    callbacks=[
        pl.callbacks.EarlyStopping(
            monitor="loss/total/val", patience=7, mode="min", verbose=True
        ),
    ],
)
trainer.fit(model, dm)
trainer.test(model, dm)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


RuntimeError: Lightning can't create new processes if CUDA is already initialized. Did you manually call `torch.cuda.*` functions, have moved the model to the device, or allocated memory on the GPU any other way? Please remove any such calls, or change the selected strategy. You will have to restart the Python kernel.